In [15]:
import pandas as pd

In [16]:
df = pd.read_csv('dataset_concat2.0.csv', sep=';')

/tmp/ipykernel_14407/2050590443.py:1: DtypeWarning: Columns (12,13,15,16,17,18,19,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('dataset_concat2.0.csv', sep=';')


In [17]:
df = df.drop(columns=['nome_autor', 'curso', 'id_projeto_pesquisa', 'id_projeto', 'id_unidade', 'id_grupo_pesquisa', 'Unnamed: 0', 'codigo_projeto', 'grupo_pesquisa'])

In [18]:
df['categoria'] = df['tipo_trabalho'].fillna('') + df['categoria_projeto'].fillna('')

In [19]:
df = df.drop(columns=['tipo_trabalho', 'categoria_projeto'])

In [20]:
df.loc[df['classificacao'] == 'pesquisa', 'nivel'] = 'pós graduação'

In [21]:
df['data_inicio'] = df['data_inicio'].combine_first(df['data_defesa'])
df['data_fim'] = df['data_fim'].combine_first(df['data_defesa'])

df = df.drop(columns=['data_defesa'])

print("\nNovas informações do DataFrame:")
df.info()

print("\nPrimeiras 5 linhas após a modificação:")
print(df[['titulo', 'data_inicio', 'data_fim']].head())


Novas informações do DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91964 entries, 0 to 91963
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   titulo                  91964 non-null  object 
 1   nome_orientador         77906 non-null  object 
 2   nivel                   91964 non-null  object 
 3   ano                     91964 non-null  float64
 4   classificacao           91964 non-null  object 
 5   unidade                 91624 non-null  object 
 6   palavras_chave          13836 non-null  object 
 7   data_inicio             91961 non-null  object 
 8   data_fim                91733 non-null  object 
 9   situacao                14058 non-null  object 
 10  tipo_projeto            14058 non-null  object 
 11  linha_pesquisa          13611 non-null  object 
 12  area_conhecimento_cnpq  13744 non-null  object 
 13  categoria               91964 non-null  object 
dtypes: fl

In [22]:
if 'unidade_x' in df.columns and 'unidade_y' in df.columns:
    df['unidade'] = df['unidade'].combine_first(df['unidade_x'])
    df['unidade'] = df['unidade'].combine_first(df['unidade_y'])

    df = df.drop(columns=['unidade_x', 'unidade_y'])

    print("Colunas de unidade ('unidade', 'unidade_x', 'unidade_y') foram juntadas.")

    print("\nNovas informações do DataFrame:")
    df.info()

In [23]:
tccs_para_alterar = df[
    (df['classificacao'] == 'tcc') &
    ((df['situacao'] != 'FINALIZADO') | (df['situacao'].isna()))
].shape[0]

df.loc[df['classificacao'] == 'tcc', 'situacao'] = 'FINALIZADO'

print("\nVerificação da situação dos TCCs após a mudança:")
print(df[df['classificacao'] == 'tcc']['situacao'].value_counts(dropna=False))


Verificação da situação dos TCCs após a mudança:


situacao
FINALIZADO    77906
Name: count, dtype: int64


In [24]:
tccs_para_alterar = df[
    (df['classificacao'] == 'tcc') &
    ((df['tipo_projeto'] != 'INTERNO') | (df['tipo_projeto'].isna()))
].shape[0]

df.loc[df['classificacao'] == 'tcc', 'tipo_projeto'] = 'INTERNO'

print("\nVerificação do 'tipo_projeto' dos TCCs após a mudança:")
print(df[df['classificacao'] == 'tcc']['tipo_projeto'].value_counts(dropna=False))


Verificação do 'tipo_projeto' dos TCCs após a mudança:
tipo_projeto
INTERNO    77906
Name: count, dtype: int64


In [25]:
df.to_csv('dataset_concat2.0_tratado_Gustavo_e_Ivan.csv', sep=';', index=False)